In [ ]:
!pip install -q transformers torch datasets accelerate tqdm bitsandbytes huggingface-hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"  
SYSTEM_PROMPT = "You are a helpful AI assistant. Be concise and friendly."

USE_HISTORY = True                
CONTEXT_METHOD = "sliding_window"

MAX_NEW_TOKENS = 256
MAX_CONTEXT_TOKENS = 2048
SLIDING_WINDOW_TURNS = 8
SUMMARY_TRIGGER_TURNS = 6
SUMMARY_MAX_NEW_TOKENS = 128

TEMPERATURE = 0.7
TOP_P = 0.9

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Using device:", device)
print("Loading tokenizer and model (may take a minute)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="auto", low_cpu_mem_usage=True)
model.eval()
print("Model loaded. Device:", model.device)
print()

def format_messages_simple(messages):
    parts = []
    for m in messages:
        parts.append(f"[{m['role'].upper()}]: {m['content'].strip()}")
    parts.append("[ASSISTANT]:")
    return "\n".join(parts)

def count_tokens_for_text(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def messages_token_count(messages):
    return count_tokens_for_text(format_messages_simple(messages))

def sliding_window_strategy(messages, keep_turns=SLIDING_WINDOW_TURNS):
    system = [m for m in messages if m["role"] == "system"]
    non_system = [m for m in messages if m["role"] != "system"]
    return system + non_system[-keep_turns:]

def token_truncate_strategy(messages, max_tokens=MAX_CONTEXT_TOKENS):
    system = [m for m in messages if m["role"] == "system"]
    non_system = [m for m in messages if m["role"] != "system"]
    kept = []
    total = count_tokens_for_text(format_messages_simple(system)) if system else 0
    for m in reversed(non_system):
        t = count_tokens_for_text(f"[{m['role'].upper()}]: {m['content']}")
        if total + t + 16 > max_tokens:  
            break
        kept.insert(0, m)
        total += t
    return system + kept

def summarize_old_strategy(messages, max_tokens=MAX_CONTEXT_TOKENS):
    system = [m for m in messages if m["role"] == "system"]
    non_system = [m for m in messages if m["role"] != "system"]
    if len(non_system) <= SUMMARY_TRIGGER_TURNS:
        return messages
    older = non_system[:-SUMMARY_TRIGGER_TURNS]
    recent = non_system[-SUMMARY_TRIGGER_TURNS:]
    older_text = "\n".join([f"[{m['role'].upper()}]: {m['content']}" for m in older])
    prompt = "Summarize the following conversation briefly and factually (few bullet points or short paragraph):\n\n" + older_text + "\n\nSummary:"
    enc = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=SUMMARY_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    summary = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if not summary:
        summary = " ".join([m["content"] for m in older])[:800]
    summary_msg = {"role": "assistant", "content": "[SUMMARY OF EARLIER CONVERSATION]: " + summary}
    new_history = system + [summary_msg] + recent
    if messages_token_count(new_history) > max_tokens:
        return token_truncate_strategy(new_history, max_tokens=max_tokens)
    return new_history

def apply_context_management(history):
    if CONTEXT_METHOD == "none":
        return history
    if CONTEXT_METHOD == "sliding_window":
        return sliding_window_strategy(history, keep_turns=SLIDING_WINDOW_TURNS)
    if CONTEXT_METHOD == "token_truncate":
        return token_truncate_strategy(history, max_tokens=MAX_CONTEXT_TOKENS)
    if CONTEXT_METHOD == "summarize_old":
        return summarize_old_strategy(history, max_tokens=MAX_CONTEXT_TOKENS)
    return history


def prepare_tokenized_input(messages):
    """
    Try tokenizer.apply_chat_template if available; otherwise fall back to manual formatting.
    Always returns dict with input_ids and attention_mask on model.device.
    """
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc["attention_mask"].to(model.device) if "attention_mask" in enc else torch.ones_like(input_ids).to(model.device)
            return {"input_ids": input_ids, "attention_mask": attention_mask}
        except Exception:
            pass
    prompt = format_messages_simple(messages)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device) if "attention_mask" in enc else torch.ones_like(input_ids).to(model.device)
    return {"input_ids": input_ids, "attention_mask": attention_mask}


def _truncate_to_first_reply(text):
    markers = ["\n[USER]:", "\n[ASSISTANT]:", "[USER]:", "[ASSISTANT]:"]
    first_pos = None
    for m in markers:
        pos = text.find(m)
        if pos != -1:
            if first_pos is None or pos < first_pos:
                first_pos = pos
    if first_pos is not None:
        return text[:first_pos].strip()
    return text.strip()

def generate_from_messages(messages, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    tensors = prepare_tokenized_input(messages)
    input_ids = tensors["input_ids"]
    attention_mask = tensors["attention_mask"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][input_ids.shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    assistant_text = _truncate_to_first_reply(raw)
    assistant_text = assistant_text.replace("[ASSISTANT]:", "").replace("[USER]:", "").strip()
    return assistant_text


print("Chat ready! Type 'exit' to stop.\n")
chat_history = [{"role": "system", "content": SYSTEM_PROMPT}]

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        break

    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit", "q"):
        print("Goodbye!")
        break

    if USE_HISTORY:
        chat_history.append({"role": "user", "content": user_input})
        active_history = apply_context_management(chat_history)
    else:
        active_history = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_input}]

    start = time.time()
    response = generate_from_messages(active_history, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE)
    latency = time.time() - start

    print("Assistant:", response)
    print(f"(Latency: {latency:.2f}s)\n")

    if USE_HISTORY:
        chat_history.append({"role": "assistant", "content": response})